# EUW Hourly Activity and Mean Game Duration

            This notebook starts with the simplest time series: how many game
            records appear in each hour, and the mean game duration in those same
            hourly bins. It is a useful first check before fitting rhythms or PCA.

## Setup

            In Colab, the notebook mounts Google Drive and looks for the raw Riot
            Parquet at the same shared-drive path used by the other release
            notebooks: `/content/drive/Shareddrives/MSc_2026_Riot/db/riotData.parquet`.

            The notebook also needs the repository Python files. If they are not
            already present in the runtime or Drive, the setup cell tries to clone
            the release repository into `/content/MSc2026_LoL_Release`.

In [ ]:
# Local users should normally use the uv environment from README.md.
# This cell only installs missing packages when the notebook is opened in Colab.
import importlib.util
import subprocess
import sys

MODULE_TO_PACKAGE = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "duckdb": "duckdb",
    "astropy": "astropy",
    "scipy": "scipy",
    "statsmodels": "statsmodels",
    "joblib": "joblib",
}

missing = [
    package
    for module, package in MODULE_TO_PACKAGE.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("Notebook packages are available.")

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore[import-not-found]  # noqa: F401
    except ImportError:
        return False
    return True


IN_COLAB = running_in_colab()
if IN_COLAB:
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")


# Override this if your repository folder has a different Colab/Drive location.
ROOT_OVERRIDE = None
REPO_URL = "https://github.com/wadelab/MSc2026_LoL_Release.git"


def find_repo_root() -> Path | None:
    if ROOT_OVERRIDE is not None:
        candidate = Path(ROOT_OVERRIDE).expanduser()
        if (candidate / "riot_analysis.py").exists():
            return candidate.resolve()
        raise FileNotFoundError(f"ROOT_OVERRIDE does not contain riot_analysis.py: {candidate}")

    candidates = list(Path.cwd().resolve().parents)
    candidates.insert(0, Path.cwd().resolve())
    candidates.extend(
        [
            Path("/content/MSc2026_LoL_Release"),
            Path("/content/drive/MyDrive/MSc2026_LoL_Release"),
            Path("/content/drive/Shareddrives/MSc_2026_Riot/MSc2026_LoL_Release"),
        ]
    )
    for candidate in candidates:
        if (candidate / "riot_analysis.py").exists():
            return candidate.resolve()
    return None


ROOT = find_repo_root()
if ROOT is None and IN_COLAB:
    clone_target = Path("/content/MSc2026_LoL_Release")
    if not clone_target.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_target)], check=True)
    ROOT = find_repo_root()

if ROOT is None:
    raise FileNotFoundError(
        "Could not find riot_analysis.py. Set ROOT_OVERRIDE to the repository folder."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 120

print(f"Repository root: {ROOT}")
print(f"Running in Colab: {IN_COLAB}")

In [ ]:
from riot_analysis import (
                AnalysisConfig,
                available_platforms,
                configure_plot_style,
                connect_analysis_database,
                filter_hourly_window,
                load_hourly_target,
                platform_overview,
                style_axes,
            )
            from server_timezones import hour_idx_to_local_hour_of_day

            configure_plot_style()

            PLATFORM = "EUW1"
            DB_FILE = ROOT / "riot_local.duckdb"
            PARQUET_FILE = None  # Auto-detects RIOT_DB_PATH, this server, or the Colab shared Drive path.
            REBUILD_HOURLY_AGG = False

            config = AnalysisConfig(
                platform=PLATFORM,
                target_col="TIMEPLAYED",
                max_hour_limit=5000,
                output_root=ROOT / "results",
            )

            conn = connect_analysis_database(
                DB_FILE,
                parquet_file=PARQUET_FILE,
                rebuild_hourly_agg=REBUILD_HOURLY_AGG,
            )

            print("Available platforms:", available_platforms(conn))
            display(platform_overview(conn).head(12))

## Load the hourly series

            `n` is the number of game records in an hourly bin. `target_mean` is
            the hourly mean of `TIMEPLAYED`, which we convert to minutes.

In [ ]:
hourly = load_hourly_target(conn, config)
            hourly = filter_hourly_window(hourly, config.max_hour_limit)
            hourly = hourly.sort_values("hour_idx").reset_index(drop=True)

            hourly["hours_since_start"] = hourly["hour_idx"] - hourly["hour_idx"].min()
            hourly["date_utc"] = pd.to_datetime(hourly["hour_idx"], unit="h", origin="unix", utc=True)
            hourly["local_hour"] = hour_idx_to_local_hour_of_day(hourly["hour_idx"], PLATFORM)
            hourly["game_records"] = hourly["n"]
            hourly["mean_duration_min"] = hourly["target_mean"] / 60.0
            hourly["game_records_7d_mean"] = hourly["game_records"].rolling(
                24 * 7,
                center=True,
                min_periods=24,
            ).mean()

            print(f"{PLATFORM}: {len(hourly):,} hourly bins after filtering")
            display(hourly.head())

## Plot activity across calendar time

            The weekly rolling line is not a model. It is only a visual guide for
            slow changes in game volume.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7.5), sharex=True)

            ax = axes[0]
            ax.plot(hourly["date_utc"], hourly["game_records"], color="#8d99ae", alpha=0.35, linewidth=0.7, label="Hourly game records")
            ax.plot(hourly["date_utc"], hourly["game_records_7d_mean"], color="#1d3557", linewidth=2.0, label="7-day rolling mean")
            ax.set_title(f"Game records per hour over time ({PLATFORM})")
            ax.set_ylabel("Game records")
            ax.legend(frameon=False)
            style_axes(ax)

            ax = axes[1]
            ax.plot(hourly["date_utc"], hourly["mean_duration_min"], color="#2a9d8f", alpha=0.85, linewidth=1.0)
            ax.set_title(f"Mean game duration per hour over time ({PLATFORM})")
            ax.set_xlabel("UTC date")
            ax.set_ylabel("Mean TIMEPLAYED (minutes)")
            style_axes(ax)

            fig.tight_layout()
            plt.show()

## Fold into the 24-hour local cycle

            Folding by local hour makes the daily pattern easier to see. The game
            duration curve uses `n` as weights so sparse hourly bins contribute
            less.

In [ ]:
volume_local = (
                hourly.groupby("local_hour", as_index=False)["game_records"]
                .mean()
                .set_index("local_hour")
                .reindex(range(24))
                .reset_index()
            )

            duration_rows = []
            for local_hour, group in hourly.groupby("local_hour"):
                duration_rows.append(
                    {
                        "local_hour": local_hour,
                        "mean_duration_min": np.average(group["mean_duration_min"], weights=group["game_records"]),
                    }
                )
            duration_local = (
                pd.DataFrame(duration_rows)
                .set_index("local_hour")
                .reindex(range(24))
                .reset_index()
            )

            fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

            ax = axes[0]
            ax.plot(volume_local["local_hour"], volume_local["game_records"], marker="o", color="#1d3557", linewidth=2.0)
            ax.set_title(f"Mean hourly game records by local hour ({PLATFORM})")
            ax.set_xlabel("Local hour")
            ax.set_ylabel("Mean game records")
            ax.set_xticks(range(0, 24, 2))
            style_axes(ax, grid_axis="y")

            ax = axes[1]
            ax.plot(duration_local["local_hour"], duration_local["mean_duration_min"], marker="o", color="#e76f51", linewidth=2.0)
            ax.set_title(f"Mean game duration by local hour ({PLATFORM})")
            ax.set_xlabel("Local hour")
            ax.set_ylabel("Mean TIMEPLAYED (minutes)")
            ax.set_xticks(range(0, 24, 2))
            style_axes(ax, grid_axis="y")

            fig.tight_layout()
            plt.show()

            display(volume_local.merge(duration_local, on="local_hour"))
            conn.close()